# BaddieVision - Single-Video Feature Extraction (Colab)

This Colab notebook extracts reusable artifacts for exactly one input video:

- `tracks.csv` for shuttle detections
- `players.csv` for the current heuristic player-activity schema
- `person_tracks.jsonl`, `pose_cache.jsonl`, and `player_assignments.jsonl` as layered player artifacts
- `metadata.json` with video and runtime details
- `preview.mp4` with shuttle and player overlays
- `<source_id>.zip` containing the full output folder

It does **not** run rally segmentation or emit rally intervals. The goal is one extraction pass on Colab so local processing can be rerun later without GPU work.


## Runtime Notes

Use **Runtime > Change runtime type > T4 GPU** (or another GPU).

Run the dependency install cell before any heavy imports. If Colab reports that imported packages were replaced, restart the runtime once and rerun from the top.

The notebook expects the private model bundle ZIP to be available in Google Drive under `MyDrive/BaddieVision/`. By default it uses one uploaded video, but it can also read a Drive path directly.
- Input videos must be `.mp4`. Full inputs are byte-copied; FFmpeg is required for keyframe-aligned stream-copy trimming and preview generation.


## 1. Configure the run

Set the input mode and optional source ID. Keep `PERSON_YOLO_MODEL` on a COCO person model such as `yolov8n.pt`; shuttle YOLO weights are only for shuttle detection and are not used for player tracking.


In [ ]:
from pathlib import Path

GITHUB_URL = "https://github.com/MaxLinCode/BaddieVision.git"
REPO_DIR = Path("/content/BaddieVision")
DRIVE_ROOT = Path("/content/drive/MyDrive/BaddieVision")

# Required model bundle in Drive.
MODELS_ZIP_NAME = "badminton-models-v1-2026-07-01.zip"

# Input mode: "upload" or "drive".
INPUT_MODE = "upload"
DRIVE_VIDEO_PATH = ""  # Example: /content/drive/MyDrive/BaddieVision/input/my_match.mp4

# Optional overrides.
SOURCE_ID = ""
USE_INPAINTNET = False  # Legacy comparison only; maintained extraction is TrackNet-only.
WORKING_FPS = 30  # Higher-FPS inputs prefer fast NVIDIA NVENC, with ultrafast CPU fallback.
START_TIME_SEC = None  # Example: 120 for 2:00 into the video.
END_TIME_SEC = None  # Stream-copy trim boundaries are keyframe-aligned and may not be frame-exact.  # Example: 360 for 6:00 into the video. Leave as None to use the rest of the file.
# COCO person detector for player tracking. Do not use the shuttle-trained YOLO here.
PERSON_YOLO_MODEL = "yolov8n.pt"
COURT_CALIBRATION = ""  # Recommended for skewed/sideways cameras; without it, player selection falls back to a rough screen-space heuristic.
# TrackNet background estimation can use a lot of RAM. Lower values reduce memory use.
TRACKNET_MAX_SAMPLE_NUM = 240
TRACKNET_VIDEO_RANGE = None  # Example: (0, 45) to estimate the median from only the first 45 seconds of the working clip.
OUTPUT_ROOT = Path("/content/colab_outputs")
COPY_ZIP_TO_DRIVE = True
DOWNLOAD_ZIP_TO_BROWSER = False

print(f"Repository target: {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Input mode: {INPUT_MODE}")


## 2. Mount Drive

Drive is used for the private model bundle and optional ZIP export.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Clone and install

This keeps the notebook independent from the current Colab filesystem state.


In [ ]:
import shutil
import subprocess

def run_git(command):
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    result.check_returncode()


def clone_fresh():
    if REPO_DIR.exists():
        print(f"Removing stale checkout at {REPO_DIR}")
        shutil.rmtree(REPO_DIR)
    run_git(["git", "clone", "--recurse-submodules", GITHUB_URL, str(REPO_DIR)])


if (REPO_DIR / ".git").is_dir():
    print("Repository already exists; updating it.")
    try:
        run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    except subprocess.CalledProcessError:
        print("Existing checkout could not be fast-forwarded; cloning a fresh copy.")
        clone_fresh()
    run_git(["git", "-C", str(REPO_DIR), "submodule", "sync", "--recursive"])
    run_git(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive"])
else:
    clone_fresh()

print(f"Repository ready at {REPO_DIR}")


In [ ]:
%cd /content/BaddieVision
%pip install -q -r requirements-colab.txt


## 4. Load imports

These imports also put the repo and TrackNet submodule on `sys.path`.


In [ ]:
import shutil
import sys
import zipfile

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

src_dir = REPO_DIR / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

tracknet_dir = src_dir / "TrackNetV3"
if str(tracknet_dir) not in sys.path:
    sys.path.insert(0, str(tracknet_dir))

tracknet_predict_args = tracknet_dir / "predictArgs.py"
predict_args_source = tracknet_predict_args.read_text(encoding="utf-8")
if "def _extend_prediction_dict(target, source):" not in predict_args_source:
    raise RuntimeError(
        "Stale TrackNetV3 checkout: src/TrackNetV3/predictArgs.py is missing the PeakValue fix. "
        "Update the TrackNetV3 submodule before running this notebook."
    )

from InPlay.heuristic.config import HeuristicConfig
from InPlay.heuristic.person_tracks import extract_person_tracks
from InPlay.heuristic.player_interpretation import interpret_players
from InPlay.heuristic.tracks import read_track_csv
from TrackNetV3.predictArgs import run_prediction
from src.pose_estimator import ensure_pose_model_asset, pose_backend_info, pose_connections
# Reload repo-local helpers so this cell also works after updating files in a live kernel.
import importlib
import src.single_video.artifacts as single_video_artifacts
import src.single_video as single_video_helpers
importlib.reload(single_video_artifacts)
importlib.reload(single_video_helpers)
from src.single_video import (
    copy_model_tree,
    load_track_models,
    looks_like_model_root,
    prepare_working_video,
    read_video_info,
    render_player_preview,
    link_shuttle_candidates,
    link_shuttle_hypotheses,
    segment_suffix,
    write_metadata,
    write_result_manifest,
    zip_result_dir,
)

POSE_MODEL_ASSET = ensure_pose_model_asset()
POSE_BACKEND = pose_backend_info(POSE_MODEL_ASSET)
print(f"Pose model asset ready: {POSE_MODEL_ASSET}")


## 5. Platform adapters

Resolve platform storage while shared video, model, preview, metadata, and ZIP helpers come from `src.single_video`.


In [ ]:
def extract_models_zip() -> None:
    archive = DRIVE_ROOT / MODELS_ZIP_NAME
    if not archive.is_file():
        raise FileNotFoundError(f"Missing model bundle: {archive}")
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(REPO_DIR)
    print(f"Extracted {archive.name} into {REPO_DIR}")
    if not looks_like_model_root(REPO_DIR, use_inpaintnet=USE_INPAINTNET):
        bundle_root = REPO_DIR / MODELS_ZIP_NAME.removesuffix(".zip")
        if not looks_like_model_root(bundle_root, use_inpaintnet=USE_INPAINTNET):
            raise FileNotFoundError(f"Extracted {archive.name}, but could not find required model files under {REPO_DIR}")
        copy_model_tree(bundle_root, REPO_DIR, use_inpaintnet=USE_INPAINTNET)


def resolve_input_video() -> Path:
    if INPUT_MODE == "drive":
        if not DRIVE_VIDEO_PATH:
            raise ValueError("Set DRIVE_VIDEO_PATH when INPUT_MODE='drive'.")
        video_path = Path(DRIVE_VIDEO_PATH)
        if not video_path.is_file():
            raise FileNotFoundError(video_path)
        return video_path
    if INPUT_MODE != "upload":
        raise ValueError("INPUT_MODE must be 'upload' or 'drive'.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Expected exactly one uploaded video, got {len(uploaded)}")
    filename = next(iter(uploaded))
    return Path("/content") / filename


## 6. Prepare output folder

This creates `colab_outputs/<source_id>/`, optionally clips the video to a requested time range, optionally downsamples it, and uses that working video for extraction.


In [ ]:
extract_models_zip()
video_path = resolve_input_video()
original_video_info = read_video_info(video_path)
clip_suffix = segment_suffix(START_TIME_SEC, END_TIME_SEC)
default_source_id = video_path.stem if clip_suffix == "full" else f"{video_path.stem}_{clip_suffix}"
source_id = SOURCE_ID or default_source_id
result_dir = OUTPUT_ROOT / source_id
result_dir.mkdir(parents=True, exist_ok=True)

working_video_path = result_dir / f"{source_id}_input.mp4"
local_video_path = prepare_working_video(video_path, working_video_path, START_TIME_SEC, END_TIME_SEC, WORKING_FPS)

print(f"Original video: {video_path}")
print(f"Working video: {local_video_path}")
print(f"Original FPS: {original_video_info['fps']}")
print(f"Source ID: {source_id}")
print(f"Result dir: {result_dir}")


## 7. Run extraction

This is the main GPU work: shuttle extraction, player extraction, validation, preview rendering, metadata, and ZIP creation. It does not run rally segmentation.


In [ ]:
video_info = read_video_info(local_video_path)
models = load_track_models(REPO_DIR, use_inpaintnet=USE_INPAINTNET)

tracks_path = result_dir / "tracks.csv"
candidates_path = result_dir / "shuttle_candidates.jsonl"
tracklets_path = result_dir / "shuttle_tracklets.jsonl"
hypotheses_path = result_dir / "shuttle_hypotheses.jsonl"
players_path = result_dir / "players.csv"
person_tracks_path = result_dir / "person_tracks.jsonl"
pose_cache_path = result_dir / "pose_cache.jsonl"
assignments_path = result_dir / "player_assignments.jsonl"
metadata_path = result_dir / "metadata.json"
preview_path = result_dir / "preview.mp4"
zip_path = result_dir / f"{source_id}.zip"
manifest_json_path = result_dir / "artifact_manifest.json"

print("[1/5] Starting TrackNet shuttle tracking...")
run_prediction(
    video_file=str(local_video_path),
    save_dir=str(result_dir),
    batch_size=16,
    output_video=False,
    large_video=True,
    max_sample_num=TRACKNET_MAX_SAMPLE_NUM,
    video_range=TRACKNET_VIDEO_RANGE,
    optimized_inference=True,
    chunk_size_frames=4096,
    num_workers=2,
    pin_memory=True,
    candidate_output_path=str(candidates_path),
    reuse_models=models,
)
print("[1/5] TrackNet shuttle tracking complete.")

tracknet_csv = result_dir / f"{local_video_path.stem.lower()}_ball.csv"
if not tracknet_csv.is_file():
    raise FileNotFoundError(f"TrackNet output missing: {tracknet_csv}")
shutil.move(tracknet_csv, tracks_path)
link_shuttle_candidates(candidates_path, tracklets_path)
print(f"[2/5] Shuttle tracks saved to {tracks_path}")

print("[3/5] Starting YOLO player tracking and MediaPipe pose extraction...")
extract_person_tracks(local_video_path, person_tracks_path, PERSON_YOLO_MODEL)
interpret_players(
    local_video_path, person_tracks_path, COURT_CALIBRATION, pose_cache_path,
    assignments_path, players_path, pose_model_asset=POSE_MODEL_ASSET,
)
print("[3/5] Player tracking and pose extraction complete.")
link_shuttle_hypotheses(candidates_path, tracklets_path, hypotheses_path)
print(f"[3/5] Shuttle association hypotheses saved to {hypotheses_path}")

print("[4/5] Validating shuttle tracks and rendering preview...")
read_track_csv(tracks_path, (video_info["width"], video_info["height"]), HeuristicConfig())
render_player_preview(local_video_path, tracks_path, person_tracks_path, assignments_path, pose_cache_path, preview_path, pose_connections(), candidate_path=candidates_path, tracklet_path=tracklets_path, hypotheses_path=hypotheses_path)
print(f"[4/5] Preview complete: {preview_path}")

print("[5/5] Writing metadata and creating ZIP bundle...")
write_metadata(
    metadata_path,
    source_id,
    video_path,
    local_video_path,
    original_video_info,
    video_info,
    models,
    START_TIME_SEC,
    END_TIME_SEC,
    WORKING_FPS,
    player_detector=PERSON_YOLO_MODEL,
    pose_backend=POSE_BACKEND,
)
write_result_manifest(manifest_json_path, result_dir, source_id, [local_video_path, tracks_path, candidates_path, tracklets_path, hypotheses_path, players_path, person_tracks_path, pose_cache_path, assignments_path, metadata_path, preview_path])
zip_result_dir(result_dir, zip_path)
print(f"[5/5] Metadata and ZIP complete: {zip_path}")

print("Artifacts:")
for path in [tracks_path, candidates_path, tracklets_path, hypotheses_path, players_path, person_tracks_path, pose_cache_path, assignments_path, metadata_path, manifest_json_path, preview_path, zip_path]:
    print(f"- {path}")


## 8. Export ZIP

Optionally copy the ZIP to Drive or download it through the browser.


In [ ]:
if COPY_ZIP_TO_DRIVE:
    drive_target = DRIVE_ROOT / "colab_outputs" / video_path.stem
    drive_target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_target / zip_path.name)
    print(f"Copied ZIP to {drive_target / zip_path.name}")

if DOWNLOAD_ZIP_TO_BROWSER:
    from google.colab import files
    files.download(str(zip_path))
